In [ ]:
# Run once in a CUDA Linux environment. mamba-ssm has platform-specific build requirements.
# %pip install -U torch mamba-ssm numpy scikit-learn pandas tqdm

from __future__ import annotations
import math, random, copy
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, cohen_kappa_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.cuda.amp import GradScaler, autocast
try:
    from mamba_ssm import Mamba
except ImportError as e:
    raise ImportError('S³-Mamba-DA requires mamba-ssm; install it in the first cell.') from e

@dataclass
class CFG:
    data_path: str = 'data/mi_eeg.npz'
    fs: int = 250
    channels: int = 22
    samples: int = 1000
    classes: int = 4
    filters: int = 10
    graph_nodes: int = 64
    d_model: int = 64
    d_state: int = 16
    batch_size: int = 64
    epochs: int = 200
    warmup_epochs: int = 10
    lr: float = 3e-4
    weight_decay: float = 1e-4
    label_smoothing: float = 0.10
    lambda_domain: float = 0.20
    lambda_supcon: float = 0.10
    temperature: float = 0.07
    val_fraction: float = 0.10
    freqmix_probability: float = 0.50
    freqmix_alpha: float = 0.40
    use_adabn: bool = True
    adabn_trials: int = 20
    seed: int = 2026

cfg = CFG()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed); random.seed(cfg.seed)
torch.backends.cudnn.benchmark = True
print('device:', device)

In [ ]:
class EEGDataset(Dataset):
    def __init__(self, x, y, subject):
        self.x = torch.as_tensor(x, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.long)
        self.subject = torch.as_tensor(subject, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i], self.subject[i]

def load_eeg_npz(path: str, cfg: CFG):
    a = np.load(path, allow_pickle=True)
    required = {'X', 'y', 'subjects'}
    if not required.issubset(a.files):
        raise KeyError(f'Expected keys {required}; found {a.files}')
    x, y, s = a['X'].astype('float32'), a['y'], a['subjects']
    if x.ndim != 3 or x.shape[1:] != (cfg.channels, cfg.samples):
        raise ValueError(f'Expected X [N,{cfg.channels},{cfg.samples}], got {x.shape}')
    # Encode arbitrary subject/class labels deterministically; preserve original values for reporting.
    y_unique, y = np.unique(y, return_inverse=True)
    s_unique, s = np.unique(s, return_inverse=True)
    if len(y_unique) != cfg.classes:
        raise ValueError(f'Expected {cfg.classes} MI classes, got {len(y_unique)}: {y_unique}')
    return EEGDataset(x, y, s), y_unique, s_unique

dataset, class_names, subject_names = load_eeg_npz(cfg.data_path, cfg)
cfg.num_subjects = len(subject_names)
print(f'{len(dataset)} trials | classes={list(class_names)} | subjects={list(subject_names)}')

In [ ]:
# ---------------------------- feature extractor ----------------------------
class RobustTrialNorm(nn.Module):
    """Median/MAD normalization independently for every trial and electrode."""
    def forward(self, x):
        median = x.median(dim=-1, keepdim=True).values
        mad = (x - median).abs().median(dim=-1, keepdim=True).values
        return (x - median) / (1.4826 * mad + 1e-6)

class SincFilterBank(nn.Module):
    """Shared, learnable, band-pass Sinc filters. Output is [B,F,C,T]."""
    def __init__(self, filters=10, fs=250, kernel_size=129, min_band_hz=2.):
        super().__init__()
        if kernel_size % 2 == 0: raise ValueError('kernel_size must be odd')
        self.filters, self.fs, self.kernel_size, self.min_band_hz = filters, fs, kernel_size, min_band_hz
        # Initialization tiles 4–40 Hz, while parameterization guarantees legal cutoffs.
        low = torch.linspace(4., 34., filters)
        band = torch.full((filters,), 6.)
        self.low_raw = nn.Parameter(torch.log(torch.expm1(low)))
        self.band_raw = nn.Parameter(torch.log(torch.expm1(band - min_band_hz)))
        n = torch.arange(-(kernel_size // 2), kernel_size // 2 + 1, dtype=torch.float32)
        self.register_buffer('n', n)
        self.register_buffer('window', torch.hamming_window(kernel_size, periodic=False))
    def cutoffs_hz(self):
        low = F.softplus(self.low_raw).clamp(max=self.fs / 2 - self.min_band_hz - 0.5)
        band = F.softplus(self.band_raw) + self.min_band_hz
        high = (low + band).clamp(max=self.fs / 2 - 0.5)
        return low, high
    def forward(self, x):
        b, c, t = x.shape
        low, high = self.cutoffs_hz()
        n = self.n.to(dtype=x.dtype); w = self.window.to(dtype=x.dtype)
        # torch.sinc(q) = sin(pi*q)/(pi*q); normalized discrete band-pass formula.
        h = (2 * high[:, None] / self.fs * torch.sinc(2 * high[:, None] * n / self.fs)
             - 2 * low[:, None] / self.fs * torch.sinc(2 * low[:, None] * n / self.fs)) * w
        h = h / (h.abs().sum(-1, keepdim=True) + 1e-8)
        z = F.conv1d(x.reshape(b * c, 1, t), h[:, None], padding=self.kernel_size // 2)
        return z.reshape(b, c, self.filters, t).permute(0, 2, 1, 3).contiguous()

class DynamicGraphSpatial(nn.Module):
    """Band-specific self-attention adjacency followed by learned C→C' node projection."""
    def __init__(self, filters, electrodes, graph_nodes, d_model, attn_dim=16):
        super().__init__()
        self.q = nn.Linear(1, attn_dim, bias=False); self.k = nn.Linear(1, attn_dim, bias=False)
        self.node_projection = nn.Linear(electrodes, graph_nodes, bias=False)
        self.band_to_d = nn.Conv1d(filters, d_model, 1, bias=False)
        self.norm = nn.BatchNorm1d(d_model)
        self.scale = attn_dim ** -0.5
    def forward(self, x, return_adjacency=False):
        # x [B,F,C,T]; descriptors retain a dynamic graph per trial and frequency band.
        desc = x.mean(-1, keepdim=True)
        q, k = self.q(desc), self.k(desc)
        a = torch.softmax(torch.matmul(q, k.transpose(-1, -2)) * self.scale, dim=-1) # [B,F,C,C]
        h = F.gelu(torch.einsum('bfij,bfjt->bfit', a, x))
        h = self.node_projection(h.transpose(-1, -2)).transpose(-1, -2)             # [B,F,C',T]
        h = h.mean(dim=2)                                                            # spatial pool: [B,F,T]
        h = self.norm(self.band_to_d(h))                                             # [B,D,T]
        return (h, a) if return_adjacency else h

class BiMamba(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.forward_mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.backward_mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.fuse = nn.Linear(2 * d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        # x [B,D,T]; Mamba uses [B,T,D]. Bidirectionality is implemented explicitly.
        u = x.transpose(1, 2)
        f = self.forward_mamba(u)
        b = torch.flip(self.backward_mamba(torch.flip(u, dims=[1])), dims=[1])
        return self.norm(self.fuse(torch.cat([f, b], dim=-1)) + u).transpose(1, 2)

class TemporalChannelAttention(nn.Module):
    def __init__(self, d_model, reduction=8):
        super().__init__()
        hidden = max(4, d_model // reduction)
        self.channel_gate = nn.Sequential(nn.Linear(d_model, hidden), nn.GELU(), nn.Linear(hidden, d_model), nn.Sigmoid())
        self.time_score = nn.Conv1d(d_model, 1, kernel_size=1)
    def forward(self, x):
        channel = self.channel_gate(x.mean(-1)).unsqueeze(-1)
        temporal = torch.softmax(self.time_score(x * channel), dim=-1)
        return (x * channel * temporal).sum(-1)  # [B,D], attentive temporal pooling

class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha): ctx.alpha = alpha; return x.view_as(x)
    @staticmethod
    def backward(ctx, grad): return -ctx.alpha * grad, None

class S3MambaDA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.normalize = RobustTrialNorm()
        self.sinc = SincFilterBank(cfg.filters, cfg.fs)
        self.dgnn = DynamicGraphSpatial(cfg.filters, cfg.channels, cfg.graph_nodes, cfg.d_model)
        self.mamba = BiMamba(cfg.d_model, cfg.d_state)
        self.attention = TemporalChannelAttention(cfg.d_model)
        self.classifier = nn.Linear(cfg.d_model, cfg.classes)
        self.domain_classifier = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model), nn.GELU(), nn.Dropout(.1), nn.Linear(cfg.d_model, cfg.num_subjects))
        self.projection = nn.Sequential(nn.Linear(cfg.d_model, 128), nn.GELU(), nn.Linear(128, 128))
    def forward(self, x, alpha_grl=0., return_graph=False):
        x = self.normalize(x)
        x = self.sinc(x)
        x = self.dgnn(x, return_adjacency=return_graph)
        if return_graph: x, adjacency = x
        x = self.mamba(x)
        z = self.attention(x)
        out = (self.classifier(z), self.domain_classifier(GradientReversal.apply(z, alpha_grl)), F.normalize(self.projection(z), dim=1))
        return (*out, adjacency) if return_graph else out

model = S3MambaDA(cfg).to(device)
print('trainable parameters:', f'{sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# ------------------------- augmentation and objectives -------------------------
def freqmix(x, y, probability=.5, alpha=.4):
    """Same-class Fourier magnitude mixing, so each hard label remains valid."""
    if random.random() >= probability: return x
    partner = torch.arange(len(y), device=x.device)
    for label in y.unique():
        ids = (y == label).nonzero(as_tuple=False).flatten()
        if len(ids) > 1: partner[ids] = ids[torch.randperm(len(ids), device=x.device)]
    spectrum, paired = torch.fft.rfft(x, dim=-1), torch.fft.rfft(x[partner], dim=-1)
    lam = float(np.random.beta(alpha, alpha))
    mixed = (lam * spectrum.abs() + (1 - lam) * paired.abs()) * torch.exp(1j * torch.angle(spectrum))
    return torch.fft.irfft(mixed, n=x.shape[-1], dim=-1)

def supervised_contrastive(z, y, subjects, temperature=.07):
    """Positive pairs: same class from *different* subjects; zero if no valid pair exists."""
    logits = z @ z.T / temperature
    logits = logits - logits.max(dim=1, keepdim=True).values.detach()
    eye = torch.eye(len(z), dtype=torch.bool, device=z.device)
    valid = ~eye
    positive = (y[:, None] == y[None, :]) & (subjects[:, None] != subjects[None, :]) & valid
    log_prob = logits - torch.logsumexp(logits.masked_fill(~valid, float('-inf')), dim=1, keepdim=True)
    n_pos = positive.sum(1)
    keep = n_pos > 0
    return -log_prob.masked_fill(~positive, 0.).sum(1)[keep].div(n_pos[keep]).mean() if keep.any() else z.new_zeros(())

def grl_schedule(epoch, total_epochs):
    p = epoch / max(1, total_epochs - 1)
    return 2 / (1 + math.exp(-10 * p)) - 1

def cosine_warmup_lambda(epoch, cfg):
    if epoch < cfg.warmup_epochs: return (epoch + 1) / cfg.warmup_epochs
    q = (epoch - cfg.warmup_epochs) / max(1, cfg.epochs - cfg.warmup_epochs)
    return .5 * (1 + math.cos(math.pi * q))

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); all_y, all_p = [], []
    for x, y, _ in loader:
        p = model(x.to(device))[0].argmax(1).cpu()
        all_y.append(y); all_p.append(p)
    y, p = torch.cat(all_y).numpy(), torch.cat(all_p).numpy()
    return {'accuracy': accuracy_score(y,p), 'balanced_accuracy': balanced_accuracy_score(y,p),
            'macro_f1': f1_score(y,p,average='macro',zero_division=0), 'kappa': cohen_kappa_score(y,p)}

def adapt_batch_norm(model, unlabeled_loader, max_trials=20):
    """AdaBN uses only target inputs; do not use labels or gradients."""
    original = model.training; model.eval()
    bns = [m for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)]
    for bn in bns: bn.train()
    seen = 0
    with torch.no_grad():
        for x, _, _ in unlabeled_loader:
            model(x.to(device))
            seen += len(x)
            if seen >= max_trials: break
    model.train(original)
    return model

In [ ]:
# ------------------------------ strict LOSO runner ------------------------------
def split_loso(dataset, held_out, val_fraction, seed):
    subjects = dataset.subject.numpy(); labels = dataset.y.numpy()
    test_idx = np.flatnonzero(subjects == held_out)
    train_pool = np.flatnonzero(subjects != held_out)
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=val_fraction, random_state=seed)
    tr_rel, va_rel = next(splitter.split(train_pool, labels[train_pool]))
    return train_pool[tr_rel], train_pool[va_rel], test_idx

def loaders_for_fold(dataset, held_out, cfg):
    tr, va, te = split_loso(dataset, held_out, cfg.val_fraction, cfg.seed + int(held_out))
    kwargs = dict(num_workers=2, pin_memory=(device.type == 'cuda'))
    return (DataLoader(Subset(dataset, tr), cfg.batch_size, shuffle=True, drop_last=True, **kwargs),
            DataLoader(Subset(dataset, va), cfg.batch_size, shuffle=False, **kwargs),
            DataLoader(Subset(dataset, te), cfg.batch_size, shuffle=False, **kwargs))

def train_fold(dataset, held_out, cfg):
    train_loader, val_loader, test_loader = loaders_for_fold(dataset, held_out, cfg)
    model = S3MambaDA(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda e: cosine_warmup_lambda(e, cfg))
    scaler = GradScaler(enabled=(device.type == 'cuda'))
    best, best_score = None, -float('inf')
    for epoch in range(cfg.epochs):
        model.train(); alpha = grl_schedule(epoch, cfg.epochs)
        totals = np.zeros(3); steps = 0
        for x, y, s in train_loader:
            x, y, s = x.to(device, non_blocking=True), y.to(device, non_blocking=True), s.to(device, non_blocking=True)
            x = freqmix(x, y, cfg.freqmix_probability, cfg.freqmix_alpha)
            optimizer.zero_grad(set_to_none=True)
            with autocast(enabled=(device.type == 'cuda')):
                logits, domain_logits, z = model(x, alpha)
                l_cls = F.cross_entropy(logits, y, label_smoothing=cfg.label_smoothing)
                l_dom = F.cross_entropy(domain_logits, s)
                l_sc = supervised_contrastive(z, y, s, cfg.temperature)
                loss = l_cls + cfg.lambda_domain * l_dom + cfg.lambda_supcon * l_sc
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            totals += [l_cls.item(), l_dom.item(), l_sc.item()]; steps += 1
        scheduler.step()
        val = evaluate(model, val_loader)
        if val['balanced_accuracy'] > best_score:
            best_score, best = val['balanced_accuracy'], copy.deepcopy(model.state_dict())
        if epoch == 0 or (epoch + 1) % 20 == 0:
            print(f'subject={held_out} epoch={epoch+1:03d} val_bal_acc={val["balanced_accuracy"]:.3f} '
                  f'losses={np.round(totals / max(steps,1), 3)}')
    model.load_state_dict(best)
    no_adaptation = evaluate(model, test_loader)
    adapted = None
    if cfg.use_adabn:
        # Copy ensures the no-adaptation result is never contaminated by test-domain statistics.
        adabn_model = copy.deepcopy(model)
        adapt_batch_norm(adabn_model, test_loader, cfg.adabn_trials)
        adapted = evaluate(adabn_model, test_loader)
    return no_adaptation, adapted

all_results = []
for held_out in range(cfg.num_subjects):
    strict, adabn = train_fold(dataset, held_out, cfg)
    row = {'subject': str(subject_names[held_out]), **{f'strict_{k}':v for k,v in strict.items()}}
    if adabn is not None: row.update({f'adabn_{k}':v for k,v in adabn.items()})
    all_results.append(row)
    display(pd.DataFrame(all_results))
results = pd.DataFrame(all_results)
display(results)
display(results.drop(columns='subject').agg(['mean', 'std']).T)
results.to_csv('s3_mamba_da_loso_results.csv', index=False)